# 🔍 Verificar Faltantes: Drive vs Supabase

Versión simplificada que solo compara Drive con Supabase

In [ ]:
# 1. INSTALACIÓN
!pip install supabase google-api-python-client google-auth-httplib2 google-auth-oauthlib pandas -q
print("✅ Librerías instaladas")

In [ ]:
# 2. IMPORTAR
import os
from google.colab import userdata, files
from supabase import create_client
from google.oauth2 import service_account
from googleapiclient.discovery import build
import pandas as pd

In [ ]:
# 3. CONEXIÓN A SUPABASE
# IMPORTANTE: En Secrets (🔑), crea 2 secretos:
# Name: SUPABASE_URL          Value: https://efxzcglhyosrxtafirhs.supabase.co
# Name: SUPABASE_ANON_KEY     Value: eyJhbGciOiJI... (el token completo)

SUPABASE_URL = userdata.get('SUPABASE_URL')
SUPABASE_KEY = userdata.get('SUPABASE_ANON_KEY')
supabase = create_client(SUPABASE_URL, SUPABASE_KEY)
print("✅ Conexión a Supabase establecida")

In [ ]:
# 4. CONEXIÓN A GOOGLE DRIVE
print("📤 Sube tu archivo de credenciales de Google (service account JSON):")
uploaded = files.upload()

CREDENTIALS_FILE = list(uploaded.keys())[0]
SCOPES = ['https://www.googleapis.com/auth/drive.readonly']
credentials = service_account.Credentials.from_service_account_file(CREDENTIALS_FILE, scopes=SCOPES)
drive_service = build('drive', 'v3', credentials=credentials)
print("✅ Conexión a Google Drive establecida")

In [ ]:
# 5. CARPETAS DE DRIVE
FOLDERS = {
    'Sala_Constitucional': {'id': '1kGbRPySacSvqZITKQ0ll4-T-O3Ns6ieN', 'tipo': 'sentencia', 'sala': 'constitucional'},
    'Sala_Penal': {'id': '1AlySvmxbfCsMV07Rs24FUt6VFQSPgPY1', 'tipo': 'sentencia', 'sala': 'penal'},
    'Sustanciacion_Constitucional': {'id': '1gAXDSFKVAzVCKAXCBVAraKMM7xCXblD8', 'tipo': 'sentencia', 'sala': 'sustanciacion'},
    'Sala_Penal_Juris': {'id': '1BL50vMykxVDiCFIQJuYlhcbvlCoD5afg', 'tipo': 'jurisprudencia', 'sala': 'penal'},
    'Sala_Constitucional_Juris': {'id': '1Ais0FpCjLPOcwOCe8I86hgbGWYj2CQFK', 'tipo': 'jurisprudencia', 'sala': 'constitucional'},
    'Sala_Plena': {'id': '1hgpAKqvge2NF-QrWDv5Lf6EKGfZJzEUo', 'tipo': 'resoluciones', 'sala': 'plena'}
}

In [ ]:
# 6. ESCANEAR DRIVE
def listar_htmls(folder_id, tipo, sala):
    archivos = []
    page_token = None
    query = f"'{folder_id}' in parents and mimeType='text/html' and trashed=false"
    
    while True:
        response = drive_service.files().list(
            q=query,
            fields='nextPageToken, files(id, name)',
            pageToken=page_token,
            pageSize=1000
        ).execute()
        
        for file in response.get('files', []):
            archivos.append({'id': file['id'], 'nombre': file['name'], 'tipo': tipo, 'sala': sala})
        
        page_token = response.get('nextPageToken')
        if not page_token:
            break
    
    return archivos

print("🔍 Escaneando HTMLs en Drive...\n")
drive_htmls = []
for nombre, config in FOLDERS.items():
    print(f"📂 {nombre}...", end=' ')
    archivos = listar_htmls(config['id'], config['tipo'], config['sala'])
    drive_htmls.extend(archivos)
    print(f"{len(archivos):,} HTMLs")

drive_ids = set([d['id'] for d in drive_htmls])
print(f"\n✅ Total en Drive: {len(drive_htmls):,}")

In [ ]:
# 7. DESCARGAR SUPABASE
print("\n🔍 Descargando metadata de Supabase...")
supabase_docs = []
offset = 0
batch_size = 1000

while True:
    response = supabase.table('documentos').select('id, tipo, sala').range(offset, offset + batch_size - 1).execute()
    if not response.data:
        break
    supabase_docs.extend(response.data)
    print(f"  Descargados: {len(supabase_docs):,}...", end='\r')
    offset += batch_size

supabase_ids = set([d['id'] for d in supabase_docs])
print(f"\n✅ Total en Supabase: {len(supabase_ids):,}")

In [ ]:
# 8. COMPARAR
print("\n" + "="*70)
print("📊 COMPARACIÓN DRIVE vs SUPABASE")
print("="*70)

falta_metadata = drive_ids - supabase_ids

print(f"\n📁 HTMLs en Drive:          {len(drive_htmls):>10,}")
print(f"💾 Documentos en Supabase:  {len(supabase_ids):>10,}")
print(f"❌ Faltantes:               {len(falta_metadata):>10,}")
print(f"📊 % Procesado:             {(len(supabase_ids)/len(drive_htmls)*100) if len(drive_htmls) > 0 else 0:>9.1f}%")

print("\n" + "="*70)

In [ ]:
# 9. DESGLOSE POR CARPETA
print("\n📂 DESGLOSE POR CARPETA:\n")

resultados = []
for nombre, config in FOLDERS.items():
    htmls_carpeta = [d for d in drive_htmls if d['tipo'] == config['tipo'] and d['sala'] == config['sala']]
    ids_carpeta = set([d['id'] for d in htmls_carpeta])
    en_supabase = len(ids_carpeta & supabase_ids)
    faltantes = len(ids_carpeta - supabase_ids)
    
    resultados.append({
        'Carpeta': nombre,
        'Total': len(htmls_carpeta),
        'En Supabase': en_supabase,
        'Faltantes': faltantes,
        '% Procesado': f"{(en_supabase/len(htmls_carpeta)*100) if len(htmls_carpeta) > 0 else 0:.1f}%"
    })

df_resultados = pd.DataFrame(resultados)
print(df_resultados.to_string(index=False))

In [ ]:
# 10. EXPORTAR CSV
if len(falta_metadata) > 0:
    print("\n💾 Exportando lista de faltantes...")
    htmls_faltantes = [d for d in drive_htmls if d['id'] in falta_metadata]
    df_faltantes = pd.DataFrame(htmls_faltantes)
    df_faltantes.to_csv('htmls_faltantes.csv', index=False)
    print(f"✅ htmls_faltantes.csv ({len(htmls_faltantes):,} registros)")
    files.download('htmls_faltantes.csv')
else:
    print("\n✅ ¡No hay HTMLs faltantes! Todo sincronizado.")